In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

# 常量参数
k1 = 4.21691E-22

k2 = 0.039590602


k3 =13.17104327
kx1=0.001007507

kx2=0.001604871
kd = 5.374494076
I = 100
Imax = 35.84931505
I0 = 0.035485714

# 反推 L 和 P 的函数
def solve_L_and_P(Y_target, n):
    def equation(L):
        x1 = np.sqrt((k2*I+kx1+kx2*k2*I+1)*(k2*I+kx1+kx2*k2*I+1)+ 8 * L * (k2*k2*k3*I*I + k1+k1*kx1*kx1+k3*k2*k2*kx2*kx2*I*I))
        F1 = np.log(L / 2) + np.log((x1 - (k2*I+kx1+kx2*k2*I+1)) / (x1 + (k2*I+kx1+kx2*k2*I+1))) + np.log(kd)
        P = 1 - (1 - 1 / (1 + np.exp(-F1)))**n
        Y = P * Imax + I0
        return Y - Y_target

    try:
        L_sol = brentq(equation, 1e-5, 100)
        # 一旦得到 L，重新计算 P
        x1 = np.sqrt((k2*I+kx1+kx2*k2*I+1)*(k2*I+kx1+kx2*k2*I+1)+ 8 * L_sol * (k2*k2*k3*I*I + k1+k1*kx1*kx1+k3*k2*k2*kx2*kx2*I*I))
        F1 = np.log(L_sol / 2) + np.log((x1 - (k2*I+kx1+kx2*k2*I+1)) / (x1 + (k2*I+kx1+kx2*k2*I+1))) + np.log(kd)
        P_sol = 1 - (1 - 1 / (1 + np.exp(-F1)))**n
        return L_sol, P_sol
    except ValueError:
        return np.nan, np.nan  # 无解

# 读取 CSV 文件（确保有列 Y 和 n）
df = pd.read_csv("E:/Desktop/operator.csv")

# 应用反推函数
results = df.apply(lambda row: solve_L_and_P(row['Y'], int(row['n'])), axis=1)

# 拆分结果为两个新列
df['L_estimated'] = [res[0] for res in results]
df['P_estimated'] = [res[1] for res in results]

# 保存结果
df.to_csv("E:/Desktop/estimated_L.csv", index=False)
print("反推完成，已保存为 Yield_with_estimated_L_P.csv")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv(r"E:/Desktop/1-4operator.csv")
n_values = range(1, 5)
LBD_values = df["LBD"].dropna().unique() # 获取 LBD 列的唯一值

# 参数初始化
I = np.logspace(-3, 2, 1000)  # 避免包含0并确保足够的数据点
k1 = 4.21691E-22
k2 = 0.039590602
k3 = 13.17104327
kx1 = 0.001007507
kx2 = 0.001604871
kd = 5.374494076
Imax=10.84931505

# 创建绘图窗口
fig, ax = plt.subplots(figsize=(6, 6))

# 为每个 n 和 LBD 组合绘制曲线和点
for n in n_values:
    for L in LBD_values:
        # 提取当前LBD和n对应的实验数据
        subset = df[(df["LBD"] == L)]
        x_subset = subset["inducer"]
        column_name = str(n)
        y_subset = subset[column_name]
        
        # 计算理论曲线
        term1 = (k2*I + kx1 + kx2*k2*I + 1)**2
        term2 = 8 * L * (k2**2*k3*I**2 + k1 + k1*kx1**2 + k3*k2**2*kx2**2*I**2)
        x1 = np.sqrt(term1 + term2)
        
        # 计算F1（修正后的公式）
        numerator = x1 - (k2*I + kx1 + kx2*k2*I + 1)
        denominator = x1 + (k2*I + kx1 + kx2*k2*I + 1)
        F1 = np.log(L/2) + np.log(numerator/denominator) + np.log(kd)
        
        # 计算P*
        P= (1 - ((1 - (1 / (1 + np.exp(-F1))))**n))
        P1 = (1 - ((1 - (1 / (1 + np.exp(-F1))))**n)) *Imax+I0
        
        # 绘制理论曲线
        ax.plot(I, P1, label=f'LBD={L:.2f}, n={n}', linewidth=1.5)
        ax.scatter(x_subset, y_subset, s=40, edgecolors='black', zorder=3)
        #ax.plot(I, P, label=f'LBD={L:.2f}, n={n}', linewidth=1.5)
        # 绘制实验数据点
        

# 坐标轴设置
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Inducer', fontsize=10)
ax.set_ylabel('RPU', fontsize=10)

ax.set_xlim(1e-2, 1e2)


# 优化图例显示
handles, labels = ax.get_legend_handles_labels()
unique_labels = dict(zip(labels, handles))  # 去重处理
#ax.legend(unique_labels.values(), unique_labels.keys(), 
#         fontsize=8, loc='upper left', bbox_to_anchor=(1, 1))
plt.legend(loc='upper left')
# 图表美化
plt.title('1-4 Operator Output', fontsize=14)
plt.grid()
plt.show()